# 07 - An AI coach with OCI Generative AI (optional)

A statistical model says *how likely* a team was to win. A language model can say *what to work on*, if you hand it the numbers and forbid it from inventing anything. This notebook:

1. trains the post-game model from notebook 01 and scores one finished game,
2. compares the player's per-minute stats with every other player in the same position (percentiles),
3. builds a prompt from those facts, and
4. sends it to an OCI Generative AI chat model, **if you configured an API key**. Without a key it stops after step 3 and prints the prompt, so the notebook runs end to end for everyone.

This is post-game analysis on purpose. Riot's policies restrict tools that give players in-game advice.

In [ ]:
import os, sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # make the `lolml` helpers importable (notebooks run from notebooks/)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from lolml import data, features as F, paths
paths.ensure_dirs()
SMOKE = bool(os.environ.get("LOLML_SMOKE"))   # LOLML_SMOKE=1 uses the small committed sample and short training budgets
SEED = 42
def save_fig(name):
    plt.tight_layout(); plt.savefig(paths.OUT / name, dpi=110); plt.show()

from sklearn.ensemble import HistGradientBoostingClassifier
from lolml import coach

In [ ]:
perf = data.load("performance")
tr, te = F.group_split(perf, 0.2, SEED)
train, test = perf.iloc[tr], perf.iloc[te]
cols = F.post_game_numeric(perf)
model = HistGradientBoostingClassifier(max_iter=100 if SMOKE else 200, random_state=SEED).fit(train[cols], train["win"])
game = test.sample(1, random_state=11).iloc[0]          # one game the model has never seen
p_win = float(model.predict_proba(game[cols].to_frame().T.astype(float))[:, 1][0])
print(f"{game['championName']} {game['teamPosition']}, {game['duration']:.0f} min, actual result: {'win' if game['win'] else 'loss'}, model: {p_win:.0%} win")

In [ ]:
report = coach.percentile_report(game, train)
pd.DataFrame(report)

In [ ]:
messages = coach.build_messages(game["championName"], game["teamPosition"], game["duration"], bool(game["win"]), p_win, report)
for m in messages:
    print(f"[{m['role']}]\n{m['content']}\n")

## Ask the model

Set `LOLML_GENAI_KEY` (an OCI Generative AI API key, see the guide), optionally `LOLML_GENAI_REGION` and `LOLML_GENAI_MODEL`, **before starting Jupyter**. The key is read from the environment and never written to the notebook.

In [ ]:
if os.environ.get("LOLML_GENAI_KEY"):
    print(coach.ask(messages))
else:
    print("LOLML_GENAI_KEY is not set: skipping the model call. The prompt above is exactly what would be sent.")